# Update Index - Risk Assessment Notebook

This notebook calculates a risk index for various countries based on crude oil supply and demand scenarios and their trade relationships with Canada.

## Methodology

The **Index** is calculated as: `Index = 100 × Shock × Exposure`

* **Shock (S)**: Measures the deviation from baseline scenarios
  * For consumption: `(forecast - baseline) / forecast`
  * For production: `(baseline - forecast) / forecast`
  
* **Exposure (E)**: Measures trade relationship strength with Canada
  * For consumption scenarios: Uses Canada's export percentage to that country
  * For production scenarios: Uses Canada's import percentage from that country
  * Source: StatsCan trade data (Feb 2021 - Feb 2026)

## Data Sources

* Production forecasts: `gold.production_forecast_table_annual`
* Consumption forecasts: `gold.consumption_forecast_table_annual`
* Trade data: Google Sheets (StatsCan exports/imports)

## Output

* **Table**: `gold.Index_table`
* **Context**: Indicates whether the risk is demand exceeding imports or supply exceeding exports
* **Index values**: Absolute values rounded to nearest whole number (minimum 1)

In [0]:
# import libraries
import pandas as pd
import numpy as np

In [0]:
#Collect annual forecast data, sum together all forecast years
df_prod = spark.sql(f"SELECT Country, generic_name as Scenario, event_direction, SUM(Heavy_crude_forecast) as forecast_sum FROM gold.production_forecast_table_annual WHERE generic_name is not NULL GROUP BY Country, generic_name, event_direction")
df_prod_pd = df_prod.toPandas()

df_cons = spark.sql(f"SELECT Country, generic_name as Scenario, event_direction, SUM(Total_crude_forecast) as forecast_sum FROM gold.consumption_forecast_table_annual WHERE generic_name is not NULL GROUP BY Country, generic_name, event_direction")
df_cons_pd = df_cons.toPandas()

df_prod_baseline = spark.sql(f"SELECT Country, Scenario, event_direction, SUM(Heavy_crude_forecast) as forecast_sum FROM gold.production_forecast_table_annual WHERE Scenario = 'baseline_scenario' GROUP BY Country, Scenario, event_direction")
df_prod_baseline_pd = df_prod_baseline.toPandas()

df_cons_baseline = spark.sql(f"SELECT Country, Scenario, event_direction, SUM(Total_crude_forecast) as forecast_sum FROM gold.consumption_forecast_table_annual WHERE Scenario = 'baseline_scenario' GROUP BY Country, Scenario, event_direction")
df_cons_baseline_pd = df_cons_baseline.toPandas()

# Concatenate both production and consumption data vertically
df = pd.concat([df_prod_pd, df_cons_pd], ignore_index=True)

#Add baselines
# For prod baseline, if event_direction is 'Production' look up Country and use the baseline for that country's production forecast
df_prod_baseline_lookup = df_prod_baseline_pd[['Country', 'forecast_sum']].rename(columns={'forecast_sum': 'baseline_prod'})
df = df.merge(df_prod_baseline_lookup, on='Country', how='left')

# For cons baseline, if event_direction is 'Consumption' look up Country and use the baseline for that country's consumption forecast
df_cons_baseline_lookup = df_cons_baseline_pd[['Country', 'forecast_sum']].rename(columns={'forecast_sum': 'baseline_cons'})
df = df.merge(df_cons_baseline_lookup, on='Country', how='left')

# Combine both baseline columns into one (production rows get prod baseline, consumption rows get cons baseline)
df['baseline'] = np.where(df['event_direction'] == 'Production', df['baseline_prod'], df['baseline_cons'])
df = df.drop(columns=['baseline_prod', 'baseline_cons'])

df.display()


Country,Scenario,event_direction,forecast_sum,baseline
Qatar,Export Cut in Qatar,Production,0.0,0.0
Venezuela,Demand Disruption in Venezuela,Production,37143.60892248385,37687.91170109102
Venezuela,Sanctions in Venezuela,Production,37782.137161523846,37687.91170109102
Kuwait,Export Cut in Kuwait,Production,9308.162108110062,9899.548823569714
Kuwait,War in Kuwait,Production,7425.432833524658,9899.548823569714
United Arab Emirates,Export Cut in United Arab Emirates,Production,null,null
Iran,Export Cut in Iran,Production,37197.68662048289,36682.921521880686
Iran,Strike in Iran,Production,31167.76676298072,36682.921521880686
Iraq,Export Cut in Iraq,Production,110524.50344780493,115497.88574132932
Iraq,War in Iraq,Production,89406.57107517107,115497.88574132932


In [0]:
# Calculate Shock (S) in a new column
# For consumption: S = (forecast_sum - baseline) / forecast_sum
# For production : S = (baseline - forecast_sum) / forecast_sum
# The variables for production are in reverse because a positive index number represents higher demand than supply
# The absolute value of the index is taken at the end and the context (supply or demand) is noted

df['Shock'] = np.where(
    df['event_direction'] == 'Consumption',
    (df['forecast_sum'] - df['baseline']) / df['forecast_sum'],  # Consumption formula
    (df['baseline'] - df['forecast_sum']) / df['forecast_sum']   # Production formula
)

df.fillna(0, inplace=True)

df.display()


Country,Scenario,event_direction,forecast_sum,baseline,Shock
Qatar,Export Cut in Qatar,Production,0.0,0.0,0.0
Venezuela,Demand Disruption in Venezuela,Production,37143.60892248385,37687.91170109102,0.014654008977509191
Venezuela,Sanctions in Venezuela,Production,37782.137161523846,37687.91170109102,-0.0024939155778827314
Kuwait,Export Cut in Kuwait,Production,9308.162108110062,9899.548823569714,0.06353420885787814
Kuwait,War in Kuwait,Production,7425.432833524658,9899.548823569714,0.3331948514670838
United Arab Emirates,Export Cut in United Arab Emirates,Production,0.0,0.0,0.0
Iran,Export Cut in Iran,Production,37197.68662048289,36682.921521880686,-0.013838632059412808
Iran,Strike in Iran,Production,31167.76676298072,36682.921521880686,0.1769505913221396
Iraq,Export Cut in Iraq,Production,110524.50344780493,115497.88574132932,0.04499800621925485
Iraq,War in Iraq,Production,89406.57107517107,115497.88574132932,0.29182770743127195


In [0]:
#Update Canada's trade relationships for Exposure (E)
# Data was obtained from StatsCan using their webapp to download import and export data for canada for the last 5 years (Feb 2021 - Feb 2026)
# https://www150.statcan.gc.ca/n1/pub/71-607-x/71-607-x2021004-eng.htm
# Imports https://docs.google.com/spreadsheets/d/1LYwuRewKYRpudNzhzupYvEyNScuHxG3amPnpgJC5z6Y/edit?gid=0#gid=0
# Exports https://docs.google.com/spreadsheets/d/1LYwuRewKYRpudNzhzupYvEyNScuHxG3amPnpgJC5z6Y/edit?gid=1809920598#gid=1809920598

sheet_id = "1LYwuRewKYRpudNzhzupYvEyNScuHxG3amPnpgJC5z6Y"
import_gid = "0" # sheet tab ID
export_gid = "1809920598"
import_url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv&gid={import_gid}"
export_url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv&gid={export_gid}"

import_df = spark.createDataFrame(pd.read_csv(import_url))
export_df = spark.createDataFrame(pd.read_csv(export_url))

# Clean column names 
def clean_column_names(df):
    for col in df.columns:
        clean_col = col.replace('(', '').replace(')', '').replace(',', '').replace(' ', '_').replace('/','per')
        if col != clean_col:
            df = df.withColumnRenamed(col, clean_col)
    return df

import_df = clean_column_names(import_df)
export_df = clean_column_names(export_df)

#Now aggregate the data for country (all years combined)
from pyspark.sql.functions import sum as spark_sum

# Aggregate imports by country (sum across all years)
import_agg = import_df \
    .groupBy("Country") \
    .agg(spark_sum("Quantity").alias("Total_Import_Quantity")) \
    .orderBy("Country")


# Aggregate exports by country (sum across all years)
export_agg = export_df \
    .groupBy("Country") \
    .agg(spark_sum("Quantity").alias("Total_Export_Quantity")) \
    .orderBy("Country")

from pyspark.sql.functions import when, sum as spark_sum

# Ensure Canada is not in either dataframe
import_agg = import_agg.filter(~(import_agg.Country == "Canada"))
export_agg = export_agg.filter(~(export_agg.Country == "Canada"))

#china and hong kong need to be combined
export_agg = export_agg.withColumn("Country", 
    when(export_agg.Country == "Hong Kong", "China")
    .when(export_agg.Country == "Hong Kong, China", "China")
    .otherwise(export_agg.Country)
)
import_agg = import_agg.withColumn("Country", 
    when(import_agg.Country == "Hong Kong", "China")
    .when(import_agg.Country == "Hong Kong, China", "China")
    .otherwise(import_agg.Country))

# Re-aggregate after renaming to combine duplicate country rows (e.g., China + Hong Kong -> China)
export_agg = export_agg.groupBy("Country").agg(spark_sum("Total_Export_Quantity").alias("Total_Export_Quantity")).orderBy("Country")
import_agg = import_agg.groupBy("Country").agg(spark_sum("Total_Import_Quantity").alias("Total_Import_Quantity")).orderBy("Country")

# Now calculate total_import_percent and total_export_percent
from pyspark.sql.functions import sum as spark_sum, col

total_import = import_agg.select(spark_sum("Total_Import_Quantity")).collect()[0][0]
total_export = export_agg.select(spark_sum("Total_Export_Quantity")).collect()[0][0]

import_agg = import_agg.withColumn("total_import_percent", col("Total_Import_Quantity") / total_import)
export_agg = export_agg.withColumn("total_export_percent", col("Total_Export_Quantity") / total_export)



In [0]:
# Create new column for Exposure (E)
# For production scenarios: use total_export_percent from export_agg
# For consumption scenarios: use total_import_percent from import_agg

# Convert Spark DataFrames to pandas for merging
import_agg_pd = import_agg.toPandas()
export_agg_pd = export_agg.toPandas()

# Merge with export_agg to get export percentages
df = df.merge(export_agg_pd[['Country', 'total_export_percent']], on='Country', how='left')

# Merge with import_agg to get import percentages
df = df.merge(import_agg_pd[['Country', 'total_import_percent']], on='Country', how='left')

# Create Exposure column: use export percent for Consumption, import percent for Production
# this is because Production in a country causes demand for Canada's exports and Consumption
    # causes supply for Canada's imports
df['Exposure'] = np.where(
    df['event_direction'] == 'Consumption',
    df['total_export_percent'],  # Consumption uses export percentage
    df['total_import_percent']   # Production uses import percentage
)

# Drop the temporary columns
df = df.drop(columns=['total_export_percent', 'total_import_percent'])

#Assign exposure of 1 for Canada
df.loc[df['Country'] == 'Canada', 'Exposure'] = 1

df.display()

Country,Scenario,event_direction,forecast_sum,baseline,Shock,Exposure
Qatar,Export Cut in Qatar,Production,0.0,0.0,0.0,null
Venezuela,Demand Disruption in Venezuela,Production,37143.60892248385,37687.91170109102,0.014654008977509191,null
Venezuela,Sanctions in Venezuela,Production,37782.137161523846,37687.91170109102,-0.0024939155778827314,null
Kuwait,Export Cut in Kuwait,Production,9308.162108110062,9899.548823569714,0.06353420885787814,null
Kuwait,War in Kuwait,Production,7425.432833524658,9899.548823569714,0.3331948514670838,null
United Arab Emirates,Export Cut in United Arab Emirates,Production,0.0,0.0,0.0,0.0
Iran,Export Cut in Iran,Production,37197.68662048289,36682.921521880686,-0.013838632059412808,null
Iran,Strike in Iran,Production,31167.76676298072,36682.921521880686,0.1769505913221396,null
Iraq,Export Cut in Iraq,Production,110524.50344780493,115497.88574132932,0.04499800621925485,null
Iraq,War in Iraq,Production,89406.57107517107,115497.88574132932,0.29182770743127195,null


In [0]:
#Finally, calculate the Index
#Index = 100 x S x E
df['Index'] = 100 * df['Shock'] * df['Exposure']

#Add in the context column
#If the number is positive, print "Greater deman relative to supply
# If the number is negative, print "Greater supply relative to demand"
# If the number is null, print "No direct impact on Canda due to absence of trade relationship"
df['Context'] = np.where(
    df['Index'] > 0,
    'Risk: Demand exceeds imports',
    np.where(
        df['Index'] < 0,
        'Risk: Supply exceeds exports',
        'No direct impact on Canada due to absence of trade relationship'
    )
)

#change the Index column to absolute value and round to the nearest whole number
df['Index'] = np.abs(df['Index']).round(0).clip(lower=1)

df.display()






Country,Scenario,event_direction,forecast_sum,baseline,Shock,Exposure,Index,Context
Qatar,Export Cut in Qatar,Production,0.0,0.0,0.0,null,null,No direct impact on Canada due to absence of trade relationship
Venezuela,Demand Disruption in Venezuela,Production,37143.60892248385,37687.91170109102,0.014654008977509191,null,null,No direct impact on Canada due to absence of trade relationship
Venezuela,Sanctions in Venezuela,Production,37782.137161523846,37687.91170109102,-0.0024939155778827314,null,null,No direct impact on Canada due to absence of trade relationship
Kuwait,Export Cut in Kuwait,Production,9308.162108110062,9899.548823569714,0.06353420885787814,null,null,No direct impact on Canada due to absence of trade relationship
Kuwait,War in Kuwait,Production,7425.432833524658,9899.548823569714,0.3331948514670838,null,null,No direct impact on Canada due to absence of trade relationship
United Arab Emirates,Export Cut in United Arab Emirates,Production,0.0,0.0,0.0,0.0,1.0,No direct impact on Canada due to absence of trade relationship
Iran,Export Cut in Iran,Production,37197.68662048289,36682.921521880686,-0.013838632059412808,null,null,No direct impact on Canada due to absence of trade relationship
Iran,Strike in Iran,Production,31167.76676298072,36682.921521880686,0.1769505913221396,null,null,No direct impact on Canada due to absence of trade relationship
Iraq,Export Cut in Iraq,Production,110524.50344780493,115497.88574132932,0.04499800621925485,null,null,No direct impact on Canada due to absence of trade relationship
Iraq,War in Iraq,Production,89406.57107517107,115497.88574132932,0.29182770743127195,null,null,No direct impact on Canada due to absence of trade relationship


In [0]:
#Write df to workspace.gold.Index_table
spark.createDataFrame(df).write.mode("overwrite").format("delta").saveAsTable(
    "gold.Index_table"
)